In [5]:
import numpy as np

DATA_PATH = r"D:\ML_Project\emotion-drift-project\data\eyet4empathy\processed"

X = np.load(DATA_PATH + "\\X_train.npy", mmap_mode='r')
y = np.load(DATA_PATH + "\\y_train.npy", mmap_mode='r')
groups = np.load(DATA_PATH + "\\groups.npy")

print("X:", X.shape)
print("y:", y.shape)
print("groups:", groups.shape)

X: (96342, 100, 38)
y: (96342,)
groups: (96342, 2)


In [6]:
# If groups are wrong or only 1 unique → fix
if len(np.unique(groups)) < 2:
    print("⚠️ Fixing groups")
    groups = np.array([i // 100 for i in range(len(y))])

# Ensure SAME length
min_len = min(len(X), len(y), len(groups))

X = X[:min_len]
y = y[:min_len]
groups = groups[:min_len]

print("✅ Final shapes:", X.shape, y.shape, groups.shape)

✅ Final shapes: (96342, 100, 38) (96342,) (96342, 2)


In [7]:
from sklearn.model_selection import GroupShuffleSplit
import numpy as np

# 🔥 FIX: Convert groups to 1D (VERY IMPORTANT)
groups = np.array([str(g[0]) + "_" + str(g[1]) for g in groups])

gss = GroupShuffleSplit(test_size=0.2, random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train = groups[train_idx]

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (74981, 100, 38) Test: (21361, 100, 38)


In [8]:
from sklearn.model_selection import GroupShuffleSplit

gss_val = GroupShuffleSplit(test_size=0.2, random_state=42)

train_idx2, val_idx = next(gss_val.split(X_train, y_train, groups_train))

X_train, X_val = X_train[train_idx2], X_train[val_idx]
y_train, y_val = y_train[train_idx2], y_train[val_idx]

print("Train:", X_train.shape, "Val:", X_val.shape)

Train: (61663, 100, 38) Val: (13318, 100, 38)


In [9]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_model(input_shape):
    model = models.Sequential([

        # CNN Block
        layers.Conv1D(32, 3, activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.3),

        layers.Conv1D(64, 3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.3),

        # LSTM
        layers.LSTM(64),
        layers.Dropout(0.3),

        # Dense
        layers.Dense(32, activation='relu'),
        layers.Dense(4, activation='softmax')   # 4 emotion classes
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

model = build_model((X_train.shape[1], X_train.shape[2]))
model.summary()

c:\Users\asus\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 100, 32)        │         3,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 100, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 50, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 50, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 50, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 50, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 25, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 25, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 45,508 (177.77 KB)

 Trainable params: 45,316 (177.02 KB)

 Non-trainable params: 192 (768.00 B)

In [10]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/30
1927/1927 ━━━━━━━━━━━━━━━━━━━━ 53s 25ms/step - accuracy: 0.9073 - loss: 0.7198 - val_accuracy: 0.9228 - val_loss: 0.4085
Epoch 2/30
1927/1927 ━━━━━━━━━━━━━━━━━━━━ 47s 24ms/step - accuracy: 0.9077 - loss: 0.4119 - val_accuracy: 0.9228 - val_loss: 0.3536
Epoch 3/30
1927/1927 ━━━━━━━━━━━━━━━━━━━━ 47s 24ms/step - accuracy: 0.9077 - loss: 0.3954 - val_accuracy: 0.9228 - val_loss: 0.3505
Epoch 4/30
1927/1927 ━━━━━━━━━━━━━━━━━━━━ 47s 24ms/step - accuracy: 0.9077 - loss: 0.3935 - val_accuracy: 0.9228 - val_loss: 0.3509
Epoch 5/30
1927/1927 ━━━━━━━━━━━━━━━━━━━━ 47s 24ms/step - accuracy: 0.9077 - loss: 0.3932 - val_accuracy: 0.9228 - val_loss: 0.3514
Epoch 6/30
1927/1927 ━━━━━━━━━━━━━━━━━━━━ 46s 24ms/step - accuracy: 0.9077 - loss: 0.3932 - val_accuracy: 0.9228 - val_loss: 0.3509
Epoch 7/30
1927/1927 ━━━━━━━━━━━━━━━━━━━━ 46s 24ms/step - accuracy: 0.9077 - loss: 0.3932 - val_accuracy: 0.9228 - val_loss: 0.3510
Epoch 8/30
1927/1927 ━━━━━━━━━━━━━━━━━━━━ 46s 24ms/step - accuracy: 0.9077 -

In [11]:
test_loss, test_acc = model.evaluate(X_test, y_test)

print("Final Test Accuracy:", test_acc)

668/668 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.9151 - loss: 0.3760
Final Test Accuracy: 0.9151256680488586
